In [3]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
import requests

print("All imports successful!")

All imports successful!


In [9]:
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
print("SSL certificates fixed!")

SSL certificates fixed!


In [11]:
import requests
import ssl
import urllib.request
import io

# Bypass SSL for the download
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

url = "https://github.com/nflverse/nflverse-data/releases/download/players/players.parquet"

print("Downloading players data...")
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, context=ctx) as response:
    data = response.read()

# Load into pandas from memory
import io
players_raw = pd.read_parquet(io.BytesIO(data))
players = players_raw[players_raw["position"].isin(POSITIONS)].copy()

print(f"✓ {len(players):,} players loaded")
players.head()

✓ 8,138 players loaded


,gsis_id,display_name,common_first_name,first_name,last_name,short_name,football_name,suffix,esb_id,nfl_id,...,status,ngs_status,ngs_status_short_description,years_of_experience,pff_position,pff_status,draft_year,draft_round,draft_pick,draft_team
1,00-0038389,Israel Abanikanda,Israel,Israel,Abanikanda,I.Abanikanda,Israel,None,ABA159567,56008,...,DEV,RES,Reserve/Future,3,HB,A,2023.0,5.0,143.0,NYJ
4,00-0031021,Jared Abbrederis,Jared,Jared,Abbrederis,J.Abbrederis,Jared,None,ABB650964,41405,...,CUT,CUT,None,4,WR,None,2014.0,5.0,176.0,GB
7,00-0032104,Ameer Abdullah,Ameer,Ameer,Abdullah,A.Abdullah,Ameer,None,ABD647726,42397,...,ACT,ACT,Active,11,HB,A,2015.0,2.0,54.0,DET
11,00-0000007,Rabih Abdullah,Rabih,Rabih,Abdullah,None,None,None,ABD675101,None,...,ACT,None,None,7,None,None,NaN,NaN,NaN,None
14,ABE498348,Walter Abercrombie,Walter,Walter,Abercrombie,None,None,None,ABE498348,None,...,ACT,None,None,7,None,None,1982.0,1.0,12.0,PIT


In [13]:
# Helper function - we'll use this for every download from here on
def download_parquet(url):
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, context=ctx) as response:
        data = response.read()
    return pd.read_parquet(io.BytesIO(data))

print("✓ Helper function ready")

✓ Helper function ready


In [17]:
players = players_raw[players_raw["position"].isin(POSITIONS)].copy()

players = players[[
    "gsis_id", "display_name", "position", "latest_team",
    "birth_date", "height", "weight", "college_name",
    "rookie_season", "draft_year", "draft_round", "draft_pick"
]].rename(columns={
    "gsis_id": "player_id",
    "display_name": "player_name",
    "latest_team": "team",
    "college_name": "college",
    "rookie_season": "rookie_year",
    "draft_year": "entry_year",
})

players.to_csv("data/player_metadata.csv", index=False)
print(f"✓ {len(players):,} players saved to data/player_metadata.csv")
players.head()

✓ 8,138 players saved to data/player_metadata.csv


,player_id,player_name,position,team,birth_date,height,weight,college,rookie_year,entry_year,draft_round,draft_pick
1,00-0038389,Israel Abanikanda,RB,DAL,2002-10-05,70.0,216.0,Pittsburgh,2023,2023.0,5.0,143.0
4,00-0031021,Jared Abbrederis,WR,DET,1990-12-17,73.0,195.0,Wisconsin,2014,2014.0,5.0,176.0
7,00-0032104,Ameer Abdullah,RB,IND,1993-06-13,69.0,203.0,Nebraska,2015,2015.0,2.0,54.0
11,00-0000007,Rabih Abdullah,RB,NE,1975-04-27,72.0,235.0,Lehigh University,1998,NaN,NaN,NaN
14,ABE498348,Walter Abercrombie,RB,PHI,1959-09-26,72.0,207.0,Baylor,1982,1982.0,1.0,12.0


In [ ]:
SEASONS = list(range(2020, 2026))

print("Downloading seasonal stats for 2020-2025 (this may take a minute)...")

seasonal_url = "https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats.parquet"
seasonal_raw = download_parquet(seasonal_url)

# Filter to skill positions only
seasonal = seasonal_raw[seasonal_raw["position"].isin(POSITIONS)].copy()

print(f"✓ {len(seasonal):,} player-season rows loaded")
print(f"  Seasons available: {sorted(seasonal['season'].unique())}")
seasonal.head()

In [ ]:
weekly = seasonal[seasonal["season"].isin(SEASONS)].copy()

# Keep only columns we need
keep_cols = [
    "player_id", "player_name", "position", "recent_team", "season", "week",
    "targets", "receptions", "receiving_yards", "receiving_tds",
    "target_share", "air_yards_share",
    "carries", "rushing_yards", "rushing_tds",
    "attempts", "completions", "passing_yards", "passing_tds",
    "interceptions", "fantasy_points", "fantasy_points_ppr"
]

# Only keep columns that actually exist in the data
keep_cols = [c for c in keep_cols if c in weekly.columns]

weekly = weekly[keep_cols].rename(columns={"recent_team": "team"})

weekly.to_csv("data/player_weekly_stats.csv", index=False)
print(f"✓ {len(weekly):,} weekly rows saved (2020-2025)")
print(f"  Seasons: {sorted(weekly['season'].unique())}")
weekly.head()

In [21]:
# Team season aggregates - the "pie" each team bakes per game
team_season = (
    weekly
    .groupby(["team", "season", "week"])
    .agg(
        team_pass_attempts=("attempts", "sum"),
        team_carries=("carries", "sum"),
        team_targets=("targets", "sum"),
    )
    .reset_index()
    .groupby(["team", "season"])
    .agg(
        games_played=("week", "nunique"),
        avg_pass_attempts_game=("team_pass_attempts", "mean"),
        avg_carries_game=("team_carries", "mean"),
        avg_targets_game=("team_targets", "mean"),
    )
    .reset_index()
)

team_season.to_csv("data/team_season_stats.csv", index=False)
print(f"✓ {len(team_season)} team-season rows saved")
team_season.head()

✓ 160 team-season rows saved


,team,season,games_played,avg_pass_attempts_game,avg_carries_game,avg_targets_game
0,ARI,2020,16,35.875000,29.937500,34.000000
1,ARI,2021,18,34.666667,28.500000,33.555556
2,ARI,2022,17,39.000000,25.529412,37.294118
3,ARI,2023,17,32.647059,27.588235,31.764706
4,ARI,2024,17,31.941176,27.235294,30.764706


In [22]:
# Per-player weekly variance - this is what drives bear/base/bull scenarios
variance_metrics = {
    "WR": ["targets", "receiving_yards", "receiving_tds", "target_share"],
    "TE": ["targets", "receiving_yards", "receiving_tds", "target_share"],
    "RB": ["carries", "rushing_yards", "rushing_tds", "targets", "receiving_yards"],
    "QB": ["attempts", "passing_yards", "passing_tds"],
}

all_variance = []

for pos, metrics in variance_metrics.items():
    pos_weekly = weekly[weekly["position"] == pos].copy()
    
    for metric in metrics:
        if metric not in pos_weekly.columns:
            continue
            
        temp = (
            pos_weekly
            .groupby(["player_id", "player_name", "season"])[metric]
            .agg(
                mean="mean",
                std="std",
                p25=lambda x: x.quantile(0.25),
                median="median",
                p75=lambda x: x.quantile(0.75),
            )
            .reset_index()
        )
        temp["metric"] = metric
        temp["position"] = pos
        all_variance.append(temp)

variance_df = pd.concat(all_variance, ignore_index=True)
variance_df.to_csv("data/player_variance.csv", index=False)
print(f"✓ {len(variance_df):,} variance rows saved")
variance_df[variance_df["player_name"] == "J.Jefferson"].head()

✓ 11,757 variance rows saved


,player_id,player_name,season,mean,std,p25,median,p75,metric,position
737,00-0036322,J.Jefferson,2020,7.812500,3.581783,4.75,8.5,11.00,targets,WR
738,00-0036322,J.Jefferson,2021,9.823529,3.046213,8.00,10.0,11.00,targets,WR
739,00-0036322,J.Jefferson,2022,10.722222,3.892762,8.00,11.0,13.00,targets,WR
740,00-0036322,J.Jefferson,2023,10.000000,3.399346,9.25,10.0,12.75,targets,WR
741,00-0036322,J.Jefferson,2024,9.000000,2.520504,8.00,8.5,9.00,targets,WR


In [23]:
print(f"weekly rows: {len(weekly):,}")
print(f"team rows: {len(team_season):,}")
print(f"players rows: {len(players):,}")

weekly rows: 27,332
team rows: 160
players rows: 8,138


In [24]:
# Seasonal summary - aggregate weekly into full season totals
seasonal_stats = (
    weekly
    .groupby(["player_id", "player_name", "position", "team", "season"])
    .agg(
        games=("week", "nunique"),
        targets=("targets", "sum"),
        receptions=("receptions", "sum"),
        receiving_yards=("receiving_yards", "sum"),
        receiving_tds=("receiving_tds", "sum"),
        target_share=("target_share", "mean"),
        carries=("carries", "sum"),
        rushing_yards=("rushing_yards", "sum"),
        rushing_tds=("rushing_tds", "sum"),
        attempts=("attempts", "sum"),
        completions=("completions", "sum"),
        passing_yards=("passing_yards", "sum"),
        passing_tds=("passing_tds", "sum"),
        interceptions=("interceptions", "sum"),
        fantasy_points=("fantasy_points", "sum"),
        fantasy_points_ppr=("fantasy_points_ppr", "sum"),
    )
    .reset_index()
)

seasonal_stats.to_csv("data/player_seasonal_stats.csv", index=False)
print(f"✓ {len(seasonal_stats):,} player-season rows saved")

# Quick sanity check - Tyreek Hill 2023
seasonal_stats[seasonal_stats["player_name"] == "T.Hill"].tail(3)


✓ 2,944 player-season rows saved


,player_id,player_name,position,team,season,games,targets,receptions,receiving_yards,receiving_tds,...,carries,rushing_yards,rushing_tds,attempts,completions,passing_yards,passing_tds,interceptions,fantasy_points,fantasy_points_ppr
764,00-0033357,T.Hill,TE,NO,2020,17,14,10,103.0,1,...,91,472.0,8,121,88,928.0,4,2.0,148.62,158.62
765,00-0033357,T.Hill,TE,NO,2023,16,40,33,291.0,2,...,81,401.0,4,11,6,83.0,1,0.0,110.52,143.52
766,00-0033357,T.Hill,TE,NO,2024,8,31,23,187.0,0,...,39,278.0,6,4,2,21.0,0,1.0,79.34,102.34


In [26]:
seasonal_stats[seasonal_stats["player_name"] == "T.Hill"].head(3)

,player_id,player_name,position,team,season,games,targets,receptions,receiving_yards,receiving_tds,...,carries,rushing_yards,rushing_tds,attempts,completions,passing_yards,passing_tds,interceptions,fantasy_points,fantasy_points_ppr
639,00-0033040,T.Hill,WR,KC,2020,18,166,111,1631.0,15,...,17,137.0,2,0,0,0.0,0,0.0,278.8,389.8
640,00-0033040,T.Hill,WR,KC,2021,20,187,134,1524.0,12,...,10,94.0,0,0,0,0.0,0,0.0,231.8,365.8
641,00-0033040,T.Hill,WR,MIA,2022,18,185,126,1779.0,7,...,9,37.0,1,0,0,0.0,0,0.0,231.6,357.6


In [27]:
# Load full names from metadata
full_names = players[["player_id", "player_name"]].rename(columns={"player_name": "full_name"})

# Join onto seasonal stats
seasonal_stats = seasonal_stats.merge(full_names, on="player_id", how="left")

# Replace short name with full name
seasonal_stats["player_name"] = seasonal_stats["full_name"].fillna(seasonal_stats["player_name"])
seasonal_stats = seasonal_stats.drop(columns=["full_name"])

# Do the same for weekly
weekly = weekly.merge(full_names, on="player_id", how="left")
weekly["player_name"] = weekly["full_name"].fillna(weekly["player_name"])
weekly = weekly.drop(columns=["full_name"])

# Resave both files
seasonal_stats.to_csv("data/player_seasonal_stats.csv", index=False)
weekly.to_csv("data/player_weekly_stats.csv", index=False)

# Sanity check - should now show Tyreek Hill
seasonal_stats[seasonal_stats["player_name"] == "Tyreek Hill"].tail(3)

,player_id,player_name,position,team,season,games,targets,receptions,receiving_yards,receiving_tds,...,carries,rushing_yards,rushing_tds,attempts,completions,passing_yards,passing_tds,interceptions,fantasy_points,fantasy_points_ppr
641,00-0033040,Tyreek Hill,WR,MIA,2022,18,185,126,1779.0,7,...,9,37.0,1,0,0,0.0,0,0.0,231.6,357.6
642,00-0033040,Tyreek Hill,WR,MIA,2023,17,179,124,1861.0,14,...,6,15.0,0,0,0,0.0,0,0.0,269.6,393.6
643,00-0033040,Tyreek Hill,WR,MIA,2024,17,122,81,959.0,6,...,8,53.0,0,0,0,0.0,0,0.0,137.2,218.2
